# 05 · Control PID dinámico del cristalizador A3110: lazo abierto → lazo cerrado → cambios de consigna

**Orden de trabajo (según lo pedido):**

| Parte | Qué se hace | Para qué |
|---|---|---|
| **1. Lazo abierto** | Se fijan las entradas manipulables en su valor nominal y se aplican perturbaciones una a una, **que suben y que bajan**, sin ruido y con ruido | Comprobar que el modelo responde con el signo y la magnitud físicos *antes* de poner control |
| **2. Lazo cerrado** | Mismas perturbaciones (con y sin ruido), con los lazos `T` (vacío, `PIRCSAHL 3101`), `V` (nivel, `LIRCSAHL 3101`) y dilución en razón (`FFIC 3102` → `FIRC 3103`) | Ver cuánto atenúa el control cada perturbación y a qué costo (variables manipuladas) |
| **3. Cambios de consigna** | Escalones y rampa en `T` y `V` (hacia arriba y hacia abajo), también con ruido en las perturbaciones; barrido del parámetro de filtro ε (IMC) y ponderación de consigna β | Ver seguimiento, sobrepaso, tiempo de asentamiento y saturación |
| **B. Agua de dilución** | Sin cambios de fondo: el análisis previo de `AIT 3105 → FFIC 3102 → FIRC 3103` | Se conserva tal cual |

**Ecuaciones de las sesiones usadas aquí** (apuntes de control, sesiones 16–22):

* PI/PID de posición: $u = \bar u + K_c\,[e + \tfrac{1}{\tau_I}\int e\,dt + \tau_D\,\dot e]$ (sesiones 16–17). Offset nulo con acción integral.
* Proceso aproximado por **FOPDT** $G(s)=K e^{-\theta s}/(\tau s+1)$ identificado con un escalón (sesión 18).
* **IMC** → PI (sesiones 21–22): $K_c=\dfrac{\tau+\theta/2}{K(\varepsilon+\theta/2)}$, $\tau_I=\tau+\theta/2$. Un solo parámetro de ajuste ε (constante de tiempo de lazo cerrado). Con θ≈0 coincide con SIMC con $\tau_c=\varepsilon$ (el que usa `autotune`).
* Cascada y *feedforward* (sesión 19): el lazo `FIRC 3103` es el esclavo rápido; la razón `FFIC 3102` compensa la perturbación medida de caudal de alimentación.
* Criterio de desempeño: IAE $=\int|e|\,dt$, sobrepaso y tiempo de asentamiento (±5 %), como en los talleres 18 y 21.

**Qué se modifica del código original.** Se agregan las partes 1–3; el código de la parte B no cambia. El controlador `PlantController` y el modelo `Plant` no se tocaron: los cambios de consigna se hacen con un envoltorio (`ConsignaVariable`) que reescribe `ctrl.sp` en cada paso.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))           # raíz del repo (donde está msmpr_core)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from msmpr_core.params import make_params, placeholders
from msmpr_core.plant import Plant, simulate
from msmpr_core import control as C
os.makedirs("../docs/fig", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

In [ ]:
p = make_params()
pl = Plant(p)
x0, u0, d0 = pl.steady_state(alpha=0.25)
m0 = pl.metrics(x0, u0, d0)
q0, qs0 = u0["Q_dil_cmd"], d0["Q_stock"]
R0 = q0 / qs0
DT = 5.0
tun, info = C.autotune(pl, x0, u0, d0, dt_s=DT)
sp = {"T": x0[1], "V": x0[0], "rho": 0.0}
print(f"Dilución nominal {3600*q0:.2f} m3/h; razón nominal R0 = {R0:.4f}; ppm nominal {m0['ppm_ClO4']:.1f}; recuperación {m0['recovery_pct']:.1f} %")
print(f"T = {x0[1]:.1f} °C, V = {x0[0]:.0f} m3, Q_out = {3600*u0['Q_out']:.0f} m3/h, m_evap = {u0['m_evap']:.3f} kg/s, tau_res = {m0['tau_h']:.2f} h")
print("Sintonía (autotune):")
for k in ("T", "V", "flow"):
    t_ = tun[k]; Ti = t_.Kp / t_.Ki if t_.Ki > 0 else np.inf
    print(f"  {k:5s} Kp = {t_.Kp:9.4g}  Ti = {Ti:8.0f} s  acción {'directa' if t_.direct else 'inversa'}")
print("Identificación FOPDT lazo T:", {k: round(float(v), 4) for k, v in info['T'].items() if k != 'unid'})

In [ ]:
# ---------------------------------------------------------------- utilidades de esta versión
class ConsignaVariable:
    # Envoltorio de PlantController: reescribe las consignas ('T', 'V') en cada paso con funciones de t.
    def __init__(self, ctrl, funcs): self.ctrl, self.funcs = ctrl, funcs
    def __call__(self, t, x, plant):
        for k, f in self.funcs.items():
            self.ctrl.sp[k] = f(t)
        return self.ctrl(t, x, plant)

def imc_pi(K, tau, theta, eps):
    # PI por IMC (sesiones 21-22). Devuelve LoopTuning.
    Kp = (tau + theta/2) / (abs(K) * (eps + theta/2)); Ti = tau + theta/2
    return C.LoopTuning(Kp=Kp, Ki=Kp/Ti, Kd=0.0, direct=bool(K < 0))

def metricas_consigna(t_s, y, y0, y1, t_step, band=0.05):
    # Sobrepaso [%], tiempo de asentamiento (+-band del salto) [h] e IAE [unid*h] de una respuesta a escalón.
    s = (y - y0) / (y1 - y0); m = t_s >= t_step
    over = 100*max(s[m].max() - 1.0, 0.0)
    fuera = np.abs(s[m] - 1.0) > band
    tset = (t_s[m][fuera][-1] - t_step)/3600 if fuera.any() else 0.0
    iae_ = np.trapezoid(np.abs(y[m] - y1), t_s[m]) / 3600
    return over, tset, iae_

def iae(t_h, e): return float(np.trapezoid(np.abs(e), t_h))

TP = 3600.0                      # instante de la perturbación o del cambio de consigna [s]
H_OL = 18*3600                   # horizonte de las perturbaciones [s]

# ---------------------------------------------------------------- ruido de las perturbaciones
class Ruido:
    # Ruido coloreado (proceso de Ornstein-Uhlenbeck, tiempo de correlación tc) en las perturbaciones de planta.
    # Representa la variabilidad real de la alimentación y de los servicios. Semilla fija: la misma realización se usa
    # en lazo abierto y cerrado y en todos los escenarios, de modo que las diferencias se deben al control y al escalón.
    SIG = {"Q_stock": ("rel", 0.015),    # caudal de alimentación: 1.5 %
           "T_feed": ("add", 0.4),       # temperatura de alimentación: 0.4 °C
           "C_stock": ("rel", 0.005),    # KNO3 en la alimentación: 0.5 %
           "CP_stock": ("rel", 0.02),    # ClO4 en la alimentación: 2 %
           "T_dil": ("add", 0.5),        # temperatura del agua de dilución: 0.5 °C
           "valve_gain": ("rel", 0.005)} # ganancia efectiva de la válvula de dilución: 0.5 %
    def __init__(self, seed=7, escala=1.0, H=30*3600, dt=DT, tc=600.0):
        rng = np.random.default_rng(seed); n = int(H/dt) + 2; phi = np.exp(-dt/tc)
        self.dt, self.n, self.s = dt, n, {}
        for k, (kind, sig) in self.SIG.items():
            e = rng.standard_normal(n); x = np.empty(n); x[0] = e[0]
            for i in range(1, n):
                x[i] = phi*x[i-1] + np.sqrt(1 - phi**2)*e[i]
            self.s[k] = escala*sig*x
    def __call__(self, d, t):
        i = min(int(t/self.dt), self.n - 1); out = dict(d)
        for k, (kind, sig) in self.SIG.items():
            base = out.get(k, 1.0) if k == "valve_gain" else out[k]
            out[k] = base*(1.0 + self.s[k][i]) if kind == "rel" else base + self.s[k][i]
        return out

RUIDO = Ruido()

# ---------------------------------------------------------------- escenarios: cada variable sube y baja
VARS = {"Q_stock":    ("Q_stock", "rel", 0.10), "T_feed": ("T alim.", "add", 5.0),
        "CP_stock":   ("ClO4 alim.", "rel", 0.25), "C_stock": ("KNO3 alim.", "rel", 0.05),
        "T_dil":      ("T agua dil.", "add", 10.0), "valve_gain": ("Válvula dil.", "rel", 0.20)}
COLV = dict(zip(VARS, ["tab:blue", "tab:red", "tab:green", "tab:purple", "tab:orange", "tab:brown"]))
ESC, COL = {}, {"Solo ruido": "k"}
for k, (lab, kind, mag) in VARS.items():
    for sg in (+1, -1):
        nombre = f"{lab} {sg*mag*100:+.0f} %" if kind == "rel" else f"{lab} {sg*mag:+.0f} °C"
        ESC[nombre] = (k, kind, mag, sg); COL[nombre] = COLV[k]
AUM = [n for n, v in ESC.items() if v[3] > 0]
DIS = [n for n, v in ESC.items() if v[3] < 0]

def hacer_dist(key, kind, mag, signo, ruido=None):
    def f(t):
        d = dict(d0)
        if t >= TP:
            d[key] = d.get(key, 1.0)*(1.0 + signo*mag) if kind == "rel" else d[key] + signo*mag
        return ruido(d, t) if ruido is not None else d
    return f

DIST = {n: hacer_dist(*v) for n, v in ESC.items()}                                   # determinista
DIST_R = {n: hacer_dist(*v, ruido=RUIDO) for n, v in ESC.items()}                    # con ruido
DIST_R["Solo ruido"] = lambda t: RUIDO(dict(d0), t)                                  # sin escalón, sólo ruido

def lazo_abierto(dist, H=H_OL, dt=DT, log=12):
    return simulate(pl, x0, H, dt, lambda t, x, pl_: u0, disturbance=dist, log_every=log)

def lazo_cerrado(dist, H=H_OL, dt=DT, log=12, tuning=tun, enabled=("T", "V", "ratio"), consigna=None):
    qsf = (lambda t: dist(t)["Q_stock"]) if dist is not None else (lambda t: qs0)    # caudal de alimentación medido (sin ruido de sensor)
    ctrl = C.PlantController(pl, u0, sp, tuning, enabled=enabled, q_stock_meas=qsf, ratio_nom=R0)
    if consigna: ctrl = ConsignaVariable(ctrl, consigna)
    return simulate(pl, x0, H, dt, ctrl, disturbance=dist, log_every=log), ctrl

# ---------------------------------------------------------------- gráficos y tablas comunes
def fig_ol(OL, titulo, archivo):
    fig, ax = plt.subplots(4, 2, figsize=(12, 11), sharex=True)
    for j, (grupo, nom) in enumerate(((AUM, "aumentos"), (DIS, "disminuciones"))):
        for n in grupo + (["Solo ruido"] if "Solo ruido" in OL else []):
            df = OL[n]; kw = dict(color=COL[n], lw=0.9 if n == "Solo ruido" else 1.4, label=n)
            ax[0, j].plot(df.t_h, df["T"] - x0[1], **kw); ax[1, j].plot(df.t_h, df.V - x0[0], **kw)
            ax[2, j].plot(df.t_h, df.ppm_ClO4, **kw); ax[3, j].plot(df.t_h, df.recovery_pct, **kw)
        ax[0, j].set(title=f"Perturbaciones: {nom}", ylabel="ΔT [°C]"); ax[0, j].legend(fontsize=7, ncol=2)
        ax[1, j].set(ylabel="ΔV [m³]"); ax[2, j].set(ylabel="ppm ClO₄⁻"); ax[2, j].axhline(p["spec_ClO4_ppm_max"], color="r", lw=1)
        ax[3, j].set(ylabel="Y [%]", xlabel="t [h]")
        for a in ax[:, j]: a.axvline(TP/3600, color="k", lw=0.6, ls=":")
    fig.suptitle(titulo, y=0.995); fig.tight_layout(); fig.savefig(archivo); plt.show()

def fig_cl(OL, CL, names, titulo, archivo):
    fig, ax = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
    for n in names:
        o, c = OL[n], CL[n]; col = COL[n]; lw = 0.9 if n == "Solo ruido" else 1.4
        ax[0, 0].plot(o.t_h, o["T"] - x0[1], color=col, ls="--", lw=0.8, alpha=0.8); ax[0, 0].plot(c.t_h, c["T"] - x0[1], color=col, lw=lw, label=n)
        ax[0, 1].plot(c.t_h, c.V - x0[0], color=col, lw=lw)
        ax[0, 2].plot(c.t_h, c.ppm_ClO4, color=col, lw=lw); ax[0, 2].plot(o.t_h, o.ppm_ClO4, color=col, ls="--", lw=0.8, alpha=0.8)
        ax[1, 0].plot(c.t_h, 100*(c.u_m_evap/u0["m_evap"] - 1), color=col, lw=lw)
        ax[1, 1].plot(c.t_h, 100*(c.u_Q_out/u0["Q_out"] - 1), color=col, lw=lw)
        ax[1, 2].plot(c.t_h, 100*(c.Qd/q0 - 1), color=col, lw=lw)
    ax[0, 0].set(ylabel="ΔT [°C]", title="T (trazo: lazo abierto; sólido: cerrado)"); ax[0, 0].legend(fontsize=7)
    ax[0, 1].set(ylabel="ΔV [m³]", title="V en lazo cerrado"); ax[0, 2].set(ylabel="ppm", title="ClO₄⁻ en producto (trazo: abierto)")
    ax[0, 2].axhline(p["spec_ClO4_ppm_max"], color="r", lw=1)
    ax[1, 0].set(xlabel="t [h]", ylabel="Δ m_evap [%]", title="Acción de control: vacío/evaporación")
    ax[1, 1].set(xlabel="t [h]", ylabel="Δ Q_out [%]", title="Acción de control: descarga")
    ax[1, 2].set(xlabel="t [h]", ylabel="Δ Q_dil real [%]", title="Agua de dilución (FIRC 3103)")
    for a in ax.ravel(): a.axvline(TP/3600, color="k", lw=0.6, ls=":")
    fig.suptitle(titulo, y=0.995); fig.tight_layout(); fig.savefig(archivo); plt.show()

def tabla_det(OL, CL, names):
    rows = []
    for n in names:
        o, c = OL[n], CL[n]
        rows.append({"perturbación": n,
                     "max|ΔT| abierto [°C]": np.abs(o["T"] - x0[1]).max(), "max|ΔT| cerrado [°C]": np.abs(c["T"] - x0[1]).max(),
                     "IAE_T abierto [°C·h]": iae(o.t_h, o["T"] - x0[1]), "IAE_T cerrado [°C·h]": iae(c.t_h, c["T"] - x0[1]),
                     "max|ΔV| abierto [m3]": np.abs(o.V - x0[0]).max(), "max|ΔV| cerrado [m3]": np.abs(c.V - x0[0]).max(),
                     "ppm final abierto": o.ppm_ClO4.iloc[-1], "ppm final cerrado": c.ppm_ClO4.iloc[-1],
                     "Y final abierto [%]": o.recovery_pct.iloc[-1], "Y final cerrado [%]": c.recovery_pct.iloc[-1]})
    return pd.DataFrame(rows).set_index("perturbación")

def sigma_ventana(df, col, t0_h=12.0):
    # desviación estándar de la señal en la ventana final, descontando la tendencia lineal (p. ej. deriva del volumen)
    w = df[df.t_h >= t0_h]; y = w[col].values; t = w.t_h.values
    return float(np.std(y - np.polyval(np.polyfit(t, y, 1), t)))

def tabla_ruido(OL, CL, names):
    rows = []
    for n in names:
        o, c = OL[n], CL[n]
        rows.append({"caso": n,
                     "σ_T abierto [°C]": sigma_ventana(o, "T"), "σ_T cerrado [°C]": sigma_ventana(c, "T"),
                     "σ_V abierto [m3]": sigma_ventana(o, "V"), "σ_V cerrado [m3]": sigma_ventana(c, "V"),
                     "σ_ppm abierto": sigma_ventana(o, "ppm_ClO4"), "σ_ppm cerrado": sigma_ventana(c, "ppm_ClO4"),
                     "σ_Y abierto [pts]": sigma_ventana(o, "recovery_pct"), "σ_Y cerrado [pts]": sigma_ventana(c, "recovery_pct"),
                     "max|ΔT| cerrado [°C]": np.abs(c["T"] - x0[1]).max()})
    return pd.DataFrame(rows).set_index("caso")

## 1. Lazo abierto: ¿el modelo responde bien a las perturbaciones?

Aquí **no hay ningún controlador**: `Q_out`, `m_evap`, `Q_dil_cmd` y `alpha` quedan en su valor nominal. Se hacen cinco verificaciones.

1. El estado nominal es estacionario (el modelo no se mueve solo).
2. La identificación FOPDT de la temperatura (la base de la sintonía) reproduce la simulación.
3. Cada perturbación mueve las variables en el sentido físico esperado, **tanto cuando la variable sube como cuando baja** (12 escenarios: caudal, temperatura y composición de la alimentación, temperatura del agua de dilución y ganancia de la válvula de dilución).
4. Dos balances comprobables a mano: la pendiente inicial del volumen y la ganancia de la válvula.
5. Lo mismo con **ruido** en las perturbaciones (sección 1.5).

Nota: sin control el volumen es un **integrador** (no tiene realimentación), de modo que una perturbación de caudal lo hace derivar sin límite, hacia arriba o hacia abajo. Eso es correcto, y es justamente lo que el lazo de nivel corrige.

In [ ]:
# 1.1  Reposo en lazo abierto con entradas y perturbaciones nominales
df0 = lazo_abierto(None, H=6*3600, log=60)
dev = {k: float(np.max(np.abs(df0[k] - df0[k].iloc[0]))) for k in ("V", "T", "C", "CP", "ppm_ClO4", "recovery_pct")}
print("Desviación máxima respecto del estado nominal en 6 h (debe ser ~0):")
print({k: f"{v:.2e}" for k, v in dev.items()})
assert max(dev.values()) < 1e-6, "el estado nominal no es estacionario"
print("OK: el estacionario se sostiene en lazo abierto.")

In [ ]:
# 1.2  Validación de la identificación FOPDT de T frente a m_evap (escalones de +10 % y de -10 % en la evaporación)
K_, tau_, th_ = info["T"]["K"], info["T"]["tau_s"], info["T"]["theta_s"]
fig, ax = plt.subplots(figsize=(6.5, 3.6)); rmses = {}
for sg, col in ((+1, "tab:blue"), (-1, "tab:red")):
    dmev = sg*0.10*u0["m_evap"]
    t_s, T_sim = C.step_test(pl, x0, u0, d0, "m_evap", dmev, lambda x, u, dd: x[1], 12*3600, dt_s=10.0)
    T_fopdt = x0[1] + K_*dmev*(1 - np.exp(-np.clip(t_s - th_, 0, None)/tau_)); rmses[sg] = float(np.sqrt(np.mean((T_sim - T_fopdt)**2)))
    ax.plot(t_s/3600, T_sim, color=col, label=f"modelo no lineal, {sg*10:+d} % m_evap"); ax.plot(t_s/3600, T_fopdt, "--", color=col, lw=0.9)
    print(f"m_evap {sg*10:+d} %: ΔT final simulado {T_sim[-1]-x0[1]:+.2f} °C | FOPDT {K_*dmev:+.2f} °C | RMSE {rmses[sg]:.3f} °C")
ax.set(xlabel="t [h]", ylabel="T [°C]", title=f"FOPDT (trazo): K={K_:.2f} °C/(kg/s), τ={tau_/3600:.2f} h, θ≈{th_:.0f} s"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("../docs/fig/05_validacion_FOPDT.png"); plt.show()

In [ ]:
# 1.3  Perturbaciones en lazo abierto, sin ruido (una a la vez, a t = 1 h): aumentos y disminuciones
OL = {n: lazo_abierto(f) for n, f in DIST.items()}
fig_ol(OL, "Lazo abierto, sin ruido", "../docs/fig/05_lazo_abierto.png")

In [ ]:
# Tabla de verificación: sentido esperado (balances y termodinámica) frente a lo simulado
signo_esp = {   # variable -> signo del efecto de SUBIR la perturbación (el de bajarla es el opuesto); 0 = sin efecto
    "Q_stock":    {"V": +1, "T": +1, "ppm_ClO4": +1},   # más alimentación tibia (55 °C > 40 °C); entra más ClO4 y la purga no cambia
    "T_feed":     {"V": 0,  "T": +1, "ppm_ClO4": 0},
    "CP_stock":   {"V": 0,  "T": 0,  "ppm_ClO4": +1},
    "C_stock":    {"V": 0,  "T": +1, "ppm_ClO4": 0},    # más soluto: más calor de cristalización a evacuar
    "T_dil":      {"V": 0,  "T": +1, "ppm_ClO4": 0},
    "valve_gain": {"V": +1, "T": -1, "ppm_ClO4": -1},   # más agua fría y más dilución del ClO4
}
tol = {"V": 1.0, "T": 0.01, "ppm_ClO4": 0.05}
rows = []
for n, (k, kind, mag, sg) in ESC.items():
    df = OL[n]; f = {"V": df.V.iloc[-1] - x0[0], "T": df["T"].iloc[-1] - x0[1], "ppm_ClO4": df.ppm_ClO4.iloc[-1] - m0["ppm_ClO4"]}
    ok = all((0 if abs(f[v]) < tol[v] else int(np.sign(f[v]))) == sg*s for v, s in signo_esp[k].items())
    rows.append({"perturbación": n, "ΔV final [m3]": f["V"], "ΔT final [°C]": f["T"], "Δppm final": f["ppm_ClO4"],
                 "ΔY final [pts]": df.recovery_pct.iloc[-1] - m0["recovery_pct"], "signo físico": "OK" if ok else "REVISAR"})
tab_ol = pd.DataFrame(rows).set_index("perturbación"); tab_ol.round(3)

In [ ]:
# 1.4  Dos balances comprobables sin simular el controlador
# (a) Al cambiar Q_stock en ΔQ, el volumen debe variar con pendiente ΔQ (nada más cambia al instante). Sube y baja.
for n, sg in (("Q_stock +10 %", +1), ("Q_stock -10 %", -1)):
    df = OL[n]; a = df[(df.t_h*3600 >= TP) & (df.t_h*3600 <= TP + 600)]
    pend = (a.V.iloc[-1] - a.V.iloc[0]) / (3600*(a.t_h.iloc[-1] - a.t_h.iloc[0]))
    print(f"(a) {n}: dV/dt inicial simulada = {pend:+.5f} m3/s | ΔQ_stock = {sg*0.10*qs0:+.5f} m3/s | error = {100*abs(pend/(sg*0.10*qs0)-1):.2f} %")
    assert abs(pend/(sg*0.10*qs0) - 1) < 0.03
# (b) Con la válvula al 80 % o al 120 %, el caudal real de dilución debe converger a 0.8 o 1.2 del comandado
for n, g in (("Válvula dil. -20 %", 0.8), ("Válvula dil. +20 %", 1.2)):
    Qd_fin = OL[n].Qd.iloc[-1]
    print(f"(b) {n}: Qd final / Qd comandado = {Qd_fin/q0:.4f} (esperado {g:.4f})")
    assert abs(Qd_fin/q0 - g) < 1e-3
print("OK: balances de volumen y de válvula consistentes, hacia arriba y hacia abajo.")

### 1.5 Ruido en las perturbaciones (lazo abierto)

En planta ninguna perturbación es un escalón limpio: el caudal, la temperatura y la composición de la alimentación fluctúan continuamente. Se modelan como **ruido coloreado** (Ornstein–Uhlenbeck, tiempo de correlación 10 min) superpuesto a cada perturbación, con estas desviaciones estándar:

| Perturbación | σ del ruido |
|---|---|
| `Q_stock` | 1.5 % |
| `T_feed` | 0.4 °C |
| `C_stock` (KNO₃) | 0.5 % |
| `CP_stock` (ClO₄⁻) | 2 % |
| `T_dil` | 0.5 °C |
| `valve_gain` | 0.5 % |

Son supuestos de trabajo: hay que reemplazarlos por la variabilidad medida en planta (historiador) cuando esté disponible. Se usa la **misma realización** de ruido en todos los escenarios y en lazo abierto y cerrado, para que las diferencias entre casos se deban sólo al escalón y al control. `Solo ruido` es el caso sin escalón. El ruido es de las perturbaciones de proceso; no se agrega ruido de medición de los sensores (salvo lo que ya tenga `PlantController`).

In [ ]:
# 1.5  Realización de ruido usada y respuesta en lazo abierto con ruido
t_h = np.arange(RUIDO.n)*RUIDO.dt/3600
fig, ax = plt.subplots(1, 6, figsize=(16, 2.6), sharex=True)
for a, (k, (kind, sig)) in zip(ax, Ruido.SIG.items()):
    a.plot(t_h[:int(6*3600/RUIDO.dt)], 100*RUIDO.s[k][:int(6*3600/RUIDO.dt)] if kind == "rel" else RUIDO.s[k][:int(6*3600/RUIDO.dt)], color=COLV[k], lw=0.8)
    a.set(title=k, xlabel="t [h]", ylabel="%" if kind == "rel" else "°C")
fig.tight_layout(); plt.show()
OL_R = {n: lazo_abierto(f) for n, f in DIST_R.items()}
fig_ol(OL_R, "Lazo abierto, con ruido en las perturbaciones", "../docs/fig/05_lazo_abierto_ruido.png")

**Lectura de la parte 1.**

* Sin control, el volumen deriva con cualquier desbalance de caudales: es un integrador puro, hacia arriba (Q_stock +10 %, válvula +20 %: +450 m³ y +48 m³ en 18 h) o hacia abajo (Q_stock −10 %, válvula −20 %: −450 m³ y −48 m³). Eso no es un error del modelo, es lo que el lazo de nivel debe evitar.
* La temperatura responde con una constante de tiempo de ~2 h (la del FOPDT), por eso el horizonte es largo. **El signo se invierte limpiamente al invertir la perturbación**: más alimentación tibia, más temperatura de alimentación o de agua de dilución, más soluto o menos agua fría de dilución suben T, y sus opuestos la bajan. El FOPDT reproduce bien tanto el escalón positivo como el negativo; la asimetría que se ve es la no linealidad.
* Una subida de T reduce la sobresaturación y con ello la recuperación; una bajada la aumenta. El efecto sobre `Y` es mucho mayor que sobre los ppm. La excepción es el soluto de alimentación, que mueve T y Y en el mismo sentido porque hay más (o menos) KNO₃ disponible.
* El ClO₄⁻ sólo mueve los ppm, y con la lentitud de la purga α = 0.25: tarda horas en llegar a su nuevo valor. Menos dilución o más alimentación también lo suben un poco.
* **Con ruido** el volumen en lazo abierto se comporta como una caminata aleatoria (integra el ruido del caudal): con sólo ruido se aleja hasta unos 15 m³ de su nominal en 18 h, mientras que T fluctúa ±0.15 °C, Y entre 10.2 y 12.1 % y los ppm entre 35.7 y 36.0. El ruido es lento (10 min), así que el proceso, que filtra con constantes de tiempo de horas, sólo atenúa sus componentes más rápidas.

## 2. Lazo cerrado: las mismas perturbaciones con el controlador

Lazos activos: `T` (`m_evap`), `V` (`Q_out`) y dilución en razón fija (`FFIC 3102` → `FIRC 3103`: la consigna de dilución es R₀ × caudal de alimentación medido). Se compara con el lazo abierto de la parte 1, primero sin ruido y luego con ruido.

In [ ]:
# 2.1  Lazo cerrado sin ruido: aumentos y disminuciones
CL = {n: lazo_cerrado(f)[0] for n, f in DIST.items()}
fig_cl(OL, CL, AUM, "Lazo cerrado, sin ruido: perturbaciones que AUMENTAN", "../docs/fig/05_lazo_cerrado_aumentos.png")
fig_cl(OL, CL, DIS, "Lazo cerrado, sin ruido: perturbaciones que DISMINUYEN", "../docs/fig/05_lazo_cerrado_disminuciones.png")
tab_cl = tabla_det(OL, CL, list(ESC)); tab_cl.round(3)

In [ ]:
# 2.2  Consistencia modelo-controlador: con acción integral, el estado final en lazo cerrado debe coincidir con el estacionario
# del mismo modelo evaluado en la perturbación (T y V en consigna, dilución = R0 * Q_stock). 24 h para que la purga converja.
rows = []
for n in ("T alim. +5 °C", "T alim. -5 °C", "KNO3 alim. +5 %", "KNO3 alim. -5 %", "Q_stock +10 %", "Q_stock -10 %", "ClO4 alim. +25 %", "ClO4 alim. -25 %"):
    f = DIST[n]; dfc, _ = lazo_cerrado(f, H=24*3600, log=60)
    d_end = f(24*3600); xs, us, ds = pl.steady_state(alpha=0.25, d=d_end, Q_dil=R0*d_end["Q_stock"]); ms = pl.metrics(xs, us, ds)
    rows.append({"perturbación": n, "ppm simulado": dfc.ppm_ClO4.iloc[-1], "ppm estacionario": ms["ppm_ClO4"],
                 "Y simulada [%]": dfc.recovery_pct.iloc[-1], "Y estacionaria [%]": ms["recovery_pct"],
                 "error relativo Y [%]": 100*abs(dfc.recovery_pct.iloc[-1]/ms["recovery_pct"] - 1)})
cons = pd.DataFrame(rows).set_index("perturbación"); print(cons.round(4).to_string())
assert cons["error relativo Y [%]"].max() < 0.1

In [ ]:
# 2.3  Lazo cerrado con ruido en las perturbaciones (misma realización de ruido que en la sección 1.5)
CL_R = {n: lazo_cerrado(f)[0] for n, f in DIST_R.items()}
fig_cl(OL_R, CL_R, AUM + ["Solo ruido"], "Lazo cerrado, con ruido: perturbaciones que AUMENTAN", "../docs/fig/05_lazo_cerrado_aumentos_ruido.png")
fig_cl(OL_R, CL_R, DIS + ["Solo ruido"], "Lazo cerrado, con ruido: perturbaciones que DISMINUYEN", "../docs/fig/05_lazo_cerrado_disminuciones_ruido.png")
print("Variabilidad en la ventana final (12-18 h, sin tendencia lineal): σ de cada variable")
tab_rui = tabla_ruido(OL_R, CL_R, ["Solo ruido"] + list(ESC)); tab_rui.round(3)

In [ ]:
# Diferencia entre lo determinista y lo ruidoso en el desempeño del lazo cerrado (IAE de T y V)
rows = []
for n in list(ESC) + ["Solo ruido"]:
    dd = CL.get(n); rr = CL_R[n]
    rows.append({"caso": n,
                 "IAE_T sin ruido [°C·h]": iae(dd.t_h, dd["T"] - x0[1]) if dd is not None else np.nan, "IAE_T con ruido [°C·h]": iae(rr.t_h, rr["T"] - x0[1]),
                 "IAE_V sin ruido [m3·h]": iae(dd.t_h, dd.V - x0[0]) if dd is not None else np.nan, "IAE_V con ruido [m3·h]": iae(rr.t_h, rr.V - x0[0]),
                 "uso de m_evap: σ [kg/s]": float(rr.u_m_evap.iloc[len(rr)//2:].std())})
pd.DataFrame(rows).set_index("caso").round(3)

**Lectura de la parte 2.**

* Los lazos `T` y `V` reducen mucho el efecto de las perturbaciones sobre la temperatura y el volumen, tanto cuando éstas suben como cuando bajan: con T alim. ±5 °C el ΔT máximo baja de ~3 a ~0.2 °C, y con Q_stock ±10 % el volumen deja de derivar (de ±450 m³ a unos pocos m³). El estado final cumple offset nulo (sesiones 16–17), como confirma la tabla de consistencia, que ahora incluye también las perturbaciones que disminuyen.
* La respuesta es simétrica en el sentido (bajar la perturbación produce la acción de control contraria), pero **no en la magnitud**: la planta es no lineal, y la acción de control, los ppm y la recuperación finales no son espejo exacto entre un aumento y una disminución.
* Los ppm de ClO₄⁻ **no** se corrigen con estos lazos: ni T ni V ni la razón fija ven el ClO₄⁻. Sólo la compensación vía `AIT 3105` (parte B) actúa sobre él.
* La perturbación de la válvula (±20 %) la absorbe `FIRC 3103` (esclavo rápido de la cascada), por eso casi no aparece en T.
* **Con ruido**, el control recorta la variabilidad de T y V pero no la de los ppm. Con sólo ruido: T pasa de ±0.17 °C (abierto) a ±0.06 °C (cerrado) y V de una caminata aleatoria que llega a −15 m³ a una banda de ±1.3 m³; la desviación estándar en la ventana final baja de 0.032 a 0.014 °C en T y de 1.6 a 0.6 m³ en V, y la de la recuperación de 0.29 a 0.23 puntos. La de los ppm queda igual (0.059 frente a 0.057), porque ninguno de estos lazos ve el ClO₄⁻. El precio es una variable manipulada que también fluctúa (σ de `m_evap` ≈ 0.03 kg/s, ~2 % del nominal).
* **IAE con ruido.** El IAE con ruido es aproximadamente el IAE sin ruido más una línea base que sólo depende del ruido (≈ 0.24 °C·h en T y ≈ 8 m³·h en V; es el caso `Solo ruido`). Por ejemplo, T alim. +5 °C: 0.265 sin ruido y 0.498 con ruido, es decir, 0.498 − 0.237 ≈ 0.26.

## 3. Cambios de consigna (seguimiento)

Se mueven las consignas de `T` y de `V` sin perturbaciones. El seguimiento se mide con sobrepaso, tiempo de asentamiento (±5 % del salto) e IAE. Además se compara la sintonía automática con la IMC de las sesiones 21–22 para distintos ε, y el efecto de la ponderación de consigna β sobre la acción de control.

In [ ]:
# 3.1  Consigna de temperatura: escalones de -2 y +2 °C y rampa de -2 °C/h entre 1 h y 2 h
H_SP = 8*3600
sp_T = {"escalón -2 °C": lambda t: x0[1] - (2.0 if t >= TP else 0.0),
        "escalón +2 °C": lambda t: x0[1] + (2.0 if t >= TP else 0.0),
        "rampa -2 °C/h (1 h)": lambda t: x0[1] - 2.0*np.clip((t - TP)/3600.0, 0.0, 1.0)}
SPT = {n: lazo_cerrado(None, H=H_SP, consigna={"T": f})[0] for n, f in sp_T.items()}
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for (n, df), col in zip(SPT.items(), ["tab:blue", "tab:red", "tab:green"]):
    ts = df.t_h.values*3600
    ax[0].plot(df.t_h, df["T"], color=col, label=n); ax[0].plot(df.t_h, [sp_T[n](t) for t in ts], color=col, ls=":", lw=0.9)
    ax[1].plot(df.t_h, df.u_m_evap, color=col)
    ax[2].plot(df.t_h, df.recovery_pct, color=col)
ax[0].set(xlabel="t [h]", ylabel="T [°C]", title="PV (sólido) y consigna (punteado)"); ax[0].legend(fontsize=7)
ax[1].set(xlabel="t [h]", ylabel="m_evap [kg/s]", title="Variable manipulada (límites 0.2x y 3x nominal)")
ax[1].axhline(3.0*u0["m_evap"], color="r", lw=0.7); ax[1].axhline(0.2*u0["m_evap"], color="r", lw=0.7)
ax[2].set(xlabel="t [h]", ylabel="Y [%]", title="Recuperación de KNO₃")
fig.tight_layout(); fig.savefig("../docs/fig/05_consigna_T.png"); plt.show()
rows = []
for n in ("escalón -2 °C", "escalón +2 °C"):
    df = SPT[n]; y1 = x0[1] + (-2.0 if "-2" in n else 2.0)
    o, ts, ia = metricas_consigna(df.t_h.values*3600, df["T"].values, x0[1], y1, TP)
    rows.append({"caso": n, "sobrepaso [%]": o, "t asentamiento ±5 % [h]": ts, "IAE [°C·h]": ia, "m_evap máx [kg/s]": df.u_m_evap.max(),
                 "m_evap mín [kg/s]": df.u_m_evap.min(), "ppm final": df.ppm_ClO4.iloc[-1], "Y final [%]": df.recovery_pct.iloc[-1]})
df = SPT["rampa -2 °C/h (1 h)"]; e = df["T"].values - np.array([sp_T["rampa -2 °C/h (1 h)"](t) for t in df.t_h.values*3600])
rows.append({"caso": "rampa -2 °C/h", "sobrepaso [%]": np.nan, "t asentamiento ±5 % [h]": np.nan, "IAE [°C·h]": iae(df.t_h, e),
             "m_evap máx [kg/s]": df.u_m_evap.max(), "m_evap mín [kg/s]": df.u_m_evap.min(), "ppm final": df.ppm_ClO4.iloc[-1], "Y final [%]": df.recovery_pct.iloc[-1]})
print(f"Error máximo de seguimiento en la rampa: {np.abs(e).max():.3f} °C")
tab_spT = pd.DataFrame(rows).set_index("caso"); tab_spT.round(3)

In [ ]:
# 3.2  Efecto de ε (IMC) y de la ponderación de consigna β sobre el escalón de -2 °C
K_, tau_, th_ = info["T"]["K"], info["T"]["tau_s"], info["T"]["theta_s"]
cand = {"autotune (SIMC, ε=0.1τ)": tun["T"]}
cand.update({f"IMC ε={f}τ": imc_pi(K_, tau_, th_, f*tau_) for f in (0.05, 0.3, 1.0)})
fT = sp_T["escalón -2 °C"]; rows = []; fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for (n, tt) in cand.items():
    df, _ = lazo_cerrado(None, H=H_SP, tuning={**tun, "T": tt}, consigna={"T": fT})
    o, ts, ia = metricas_consigna(df.t_h.values*3600, df["T"].values, x0[1], x0[1] - 2.0, TP)
    ax[0].plot(df.t_h, df["T"], label=n); ax[1].plot(df.t_h, df.u_m_evap)
    rows.append({"sintonía": n, "Kp": tt.Kp, "Ti [s]": tt.Kp/tt.Ki, "sobrepaso [%]": o, "t asent. [h]": ts, "IAE [°C·h]": ia, "m_evap máx [kg/s]": df.u_m_evap.max()})
ax[0].plot(df.t_h, [fT(t) for t in df.t_h*3600], "k:", lw=0.9, label="consigna"); ax[0].legend(fontsize=7)
ax[0].set(xlabel="t [h]", ylabel="T [°C]", title="Escalón de consigna -2 °C: barrido de ε")
ax[1].axhline(3.0*u0["m_evap"], color="r", lw=0.7); ax[1].set(xlabel="t [h]", ylabel="m_evap [kg/s]", title="Variable manipulada (saturación en rojo)")
fig.tight_layout(); fig.savefig("../docs/fig/05_barrido_epsilon.png"); plt.show()
tab_eps = pd.DataFrame(rows).set_index("sintonía"); display(tab_eps.round(3)) if "display" in globals() else print(tab_eps.round(3))

rows = []
for beta in (1.0, 0.5, 0.0):
    qsf = lambda t: qs0
    ctrl = C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "ratio"), q_stock_meas=qsf, ratio_nom=R0); ctrl.pid_T.beta = beta
    ctrl.pid_T.I = -ctrl.pid_T.Kp*ctrl.pid_T.s*(beta*x0[1] - x0[1])      # arranque sin salto: con β≠1, P(SP=PV) = Kp(β-1)SP ≠ 0
    df = simulate(pl, x0, H_SP, DT, ConsignaVariable(ctrl, {"T": fT}), log_every=12)
    o, ts, ia = metricas_consigna(df.t_h.values*3600, df["T"].values, x0[1], x0[1] - 2.0, TP)
    rows.append({"β": beta, "sobrepaso [%]": o, "t asent. [h]": ts, "IAE [°C·h]": ia, "m_evap máx [kg/s]": df.u_m_evap.max()})
print("Ponderación de consigna β (P actúa sobre β·SP - PV; el integral siempre sobre SP - PV). El integral se precarga para que el estacionario inicial no se perturbe:")
pd.DataFrame(rows).set_index("β").round(3)

In [ ]:
# 3.3  Consigna de nivel: escalones de ±20 m3 (±3 %) y rampa de +20 m3 en 1 h
sp_V = {"escalón +20 m3": lambda t: x0[0] + (20.0 if t >= TP else 0.0),
        "escalón -20 m3": lambda t: x0[0] - (20.0 if t >= TP else 0.0),
        "rampa +20 m3/h": lambda t: x0[0] + 20.0*np.clip((t - TP)/3600.0, 0.0, 1.0)}
SPV = {n: lazo_cerrado(None, H=H_SP, consigna={"V": f})[0] for n, f in sp_V.items()}
fig, ax = plt.subplots(1, 3, figsize=(15, 4)); rows = []
for (n, df), col in zip(SPV.items(), ["tab:blue", "tab:red", "tab:green"]):
    ts = df.t_h.values*3600; ref = np.array([sp_V[n](t) for t in ts])
    ax[0].plot(df.t_h, df.V, color=col, label=n); ax[0].plot(df.t_h, ref, color=col, ls=":", lw=0.9)
    ax[1].plot(df.t_h, df.u_Q_out*3600, color=col); ax[2].plot(df.t_h, df["T"] - x0[1], color=col)
    if "escalón" in n:
        o, tset, ia = metricas_consigna(ts, df.V.values, x0[0], ref[-1], TP)
    else:
        o, tset, ia = np.nan, np.nan, iae(df.t_h, df.V.values - ref)
    rows.append({"caso": n, "sobrepaso [%]": o, "t asent. [h]": tset, "IAE [m3·h]": ia, "Q_out mín [m3/h]": 3600*df.u_Q_out.min(),
                 "Q_out máx [m3/h]": 3600*df.u_Q_out.max(), "max|ΔT| [°C]": np.abs(df["T"] - x0[1]).max()})
ax[0].set(xlabel="t [h]", ylabel="V [m³]", title="Nivel: PV (sólido) y consigna (punteado)"); ax[0].legend(fontsize=7)
ax[1].set(xlabel="t [h]", ylabel="Q_out [m³/h]", title="Variable manipulada"); ax[2].set(xlabel="t [h]", ylabel="ΔT [°C]", title="Interacción sobre la temperatura")
fig.tight_layout(); fig.savefig("../docs/fig/05_consigna_V.png"); plt.show()
tab_spV = pd.DataFrame(rows).set_index("caso"); tab_spV.round(3)

**Lectura de la parte 3.**

* **Temperatura.** Ambos escalones saturan la variable manipulada un rato: el de -2 °C llega al máximo (3× nominal) y el de +2 °C al mínimo (0.2× nominal), porque el término proporcional entrega de golpe Kp·ΔSP. El anti-windup por retrocálculo evita que el integral se descontrole, pero el sobrepaso (≈5 % y ≈2 %) y el tiempo de asentamiento sí se resienten. La respuesta es asimétrica (≈2.0 h para -2 °C frente a ≈0.5 h para +2 °C): es el efecto de la no linealidad del proceso (la cinética y el calor de cristalización dependen de T).
* **Barrido de ε.** ε chico (lazo rápido) da más ganancia, menor IAE y más esfuerzo de control (con ε = 0.05τ el actuador sigue saturando); ε = 0.3τ ya no satura y ε = τ no tiene sobrepaso pero tarda ≈6 h. Es el compromiso de la sesión 21: un solo parámetro para negociar velocidad contra robustez y uso del actuador. La sintonía automática usa SIMC con Ti = 4τ_c (más acción integral que IMC con Ti = τ), lo que explica su sobrepaso algo mayor.
* **Ponderación β.** Con β = 0.5 el golpe proporcional se reduce a la mitad: el actuador llega sólo a ≈3.3 kg/s (sin saturar), el sobrepaso baja a <1 % y el IAE sube. Con β = 0 el actuador apenas se mueve al principio y la respuesta es mucho más lenta. Detalle de implementación: `PID` arranca con integral nulo, de modo que con β ≠ 1 hay que precargarlo (como se hace en el código) para que el estacionario inicial no se perturbe.
* **Rampa.** Un PI sigue una rampa con error de seguimiento no nulo mientras dura la rampa (el error máximo está en la salida) y lo elimina después, como corresponde a un lazo tipo 1.
* **Nivel.** El lazo `V` es integrador con τ_c = 600 s y presenta sobrepaso de ~13 % ante escalones de consigna (típico de PI sobre integrador con Ti = 4τ_c). Para evitarlo en planta conviene una rampa de consigna o β < 1 en el nivel. La interacción sobre T es mínima (centésimas de °C).
* Cambiar las consignas de T o V cambia el punto de operación: la recuperación y los ppm finales de la tabla ya no son los nominales, y eso se debe analizar con la economía del proceso, no sólo con el desempeño del lazo.

In [ ]:
# 3.4  Seguimiento de consigna con ruido en las perturbaciones (misma realización de ruido): T -2 °C y +2 °C, V +20 m3 y -20 m3
rows = []; fig, ax = plt.subplots(2, 2, figsize=(13, 7))
casos_sp = [("T", "escalón -2 °C", sp_T["escalón -2 °C"], x0[1] - 2.0), ("T", "escalón +2 °C", sp_T["escalón +2 °C"], x0[1] + 2.0),
            ("V", "escalón +20 m3", sp_V["escalón +20 m3"], x0[0] + 20.0), ("V", "escalón -20 m3", sp_V["escalón -20 m3"], x0[0] - 20.0)]
for a_, (var, n, fsp, y1) in zip(ax.ravel(), casos_sp):
    sin = SPT[n] if var == "T" else SPV[n]
    con, _ = lazo_cerrado(DIST_R["Solo ruido"], H=H_SP, consigna={var: fsp})
    y0 = x0[1] if var == "T" else x0[0]; col = "T" if var == "T" else "V"
    for lab, df, c_ in (("sin ruido", sin, "tab:blue"), ("con ruido", con, "tab:red")):
        a_.plot(df.t_h, df[col], color=c_, lw=1.2, label=lab)
        o, ts, ia = metricas_consigna(df.t_h.values*3600, df[col].values, y0, y1, TP, band=0.05 if var == "V" else 0.10)
        rows.append({"consigna": f"{var} {n}", "caso": lab, "sobrepaso [%]": o, "t asent. [h]": ts, "IAE": ia,
                     "σ de la PV en 5-8 h": float(df[df.t_h >= 5.0][col].std()), "Y final [%]": df.recovery_pct.iloc[-1]})
    a_.plot(sin.t_h, [fsp(t) for t in sin.t_h*3600], "k:", lw=0.9, label="consigna"); a_.set(xlabel="t [h]", ylabel=col, title=f"{var}: {n}"); a_.legend(fontsize=7)
fig.tight_layout(); fig.savefig("../docs/fig/05_consigna_ruido.png"); plt.show()
print("Banda de asentamiento: ±10 % del salto en T (±0.2 °C) y ±5 % en V; con ruido la PV puede entrar y salir de la banda.")
tab_spR = pd.DataFrame(rows).set_index(["consigna", "caso"]); tab_spR.round(3)

**Lectura de 3.4.** El ruido de las perturbaciones se superpone al transitorio de la consigna.
* **Temperatura:** el sobrepaso y el IAE casi no cambian (IAE 0.51 → 0.50 °C·h en −2 °C; 0.58 → 0.61 en +2 °C), porque el ruido de T es pequeño frente al salto de 2 °C. La dispersión de la PV después del transitorio es ~0.016 °C.
* **Nivel:** el ruido pesa más. El sobrepaso pasa de 13.5 % a ≈16 % (+20 m³) y a ≈12 % (−20 m³), el IAE sube de 4.9 a 7.2 y 6.1 m³·h, y la PV queda oscilando ≈0.47 m³ alrededor de la consigna, porque el lazo de nivel tiene que absorber el ruido del caudal de alimentación.
* Con ruido el tiempo de asentamiento es menos nítido (la PV entra y sale de la banda); por eso en T la banda se amplió a ±10 % del salto en esta tabla. Los resultados de 3.1–3.3 usan ±5 %, y no son comparables numéricamente con ésta.

---
# Parte B · Agua de dilución: `AIT 3105 → FFIC 3102 → FIRC 3103` (sin cambios)
El resto del notebook es el análisis original, que no se modificó salvo la numeración de los títulos y la reutilización de las variables definidas arriba (`p`, `pl`, `x0`, `u0`, `d0`, `q0`, `qs0`, `R0`, `DT`, `tun`, `sp`).

## B1. ¿Cuánta agua haría falta para compensar un aumento de ClO₄⁻? (estático)
Para mantener la mezcla en su concentración nominal, la dilución debe crecer con el ClO₄⁻ de entrada. Se resuelve el estacionario con y sin esa compensación.

In [ ]:
rows = []
for f in (1.05, 1.10, 1.25, 1.50):
    R = f*(1 + R0) - 1; Qd = R*qs0
    d = {**d0, "CP_stock": f*d0["CP_stock"]}
    xs, us, ds = pl.steady_state(alpha=0.25, d=d); ms = pl.metrics(xs, us, ds)                   # sin compensar
    xc, uc, dc = pl.steady_state(alpha=0.25, d=d, Q_dil=Qd); mc = pl.metrics(xc, uc, dc)       # con compensación total
    rows.append({"ClO4 de entrada x": f, "dilución necesaria [m3/h]": 3600*Qd, "veces la nominal": Qd/q0,
                 "ppm sin comp.": ms["ppm_ClO4"], "Y sin comp. %": ms["recovery_pct"],
                 "ppm con comp.": mc["ppm_ClO4"], "Y con comp. %": mc["recovery_pct"]})
req = pd.DataFrame(rows).set_index("ClO4 de entrada x"); req.round(2)

**Lectura.** Compensar el ClO₄⁻ con agua es caro: la dilución necesaria crece varias veces sobre la nominal y la recuperación de KNO₃ cae rápido, porque el agua baja la concentración de la alimentación hacia la solubilidad. Con los placeholders actuales (alimentación apenas sobre la solubilidad supuesta) el efecto es extremo; la **tendencia** (recuperación contra pureza) es lo que importa, y el tamaño de la válvula de dilución pasa a ser una restricción de diseño.

## B2. Etapa 1 · `FIRC 3103` solo
Consigna +15 % a los 10 min y pérdida de 20 % de ganancia de la válvula a los 30 min (ruido de medición 1 %). Muestreo de 2 s.

In [ ]:
tun2, _ = C.autotune(pl, x0, u0, d0, dt_s=2.0)
flow_sp = lambda t: q0*(1.15 if t >= 600 else 1.0)
dist1 = lambda t: {**d0, "valve_gain": 0.8 if t >= 1800 else 1.0}
cl = simulate(pl, x0, 3600, 2.0, C.PlantController(pl, u0, sp, tun2, enabled=("flow",), flow_sp=flow_sp, noise_rel={"flow": 0.01}, seed=1), disturbance=dist1, log_every=5)
ol = simulate(pl, x0, 3600, 2.0, lambda t, x, pl_: {**u0, "Q_dil_cmd": flow_sp(t)}, disturbance=dist1, log_every=5)
late = lambda df: 100*abs(df.Qd.values[df.t_h.values > 0.9].mean()/(1.15*q0) - 1)
print(f"Error final tras la perturbación de válvula: FIRC 3103 {late(cl):.2f} %  |  válvula manual {late(ol):.2f} %")

## B3. Etapa 2 · `FFIC 3102`: la dilución sigue al caudal de alimentación
La alimentación sube +12 % y +6 °C a las 2 h. Con razón fija, el caudal de dilución sube con ella.

In [ ]:
Qs = lambda t: qs0*(1.12 if t >= 2*3600 else 1.0)
dist2 = lambda t: {**d0, "Q_stock": Qs(t), "T_feed": d0["T_feed"] + (6.0 if t >= 2*3600 else 0.0)}
H = 8*3600
fija = simulate(pl, x0, H, DT, C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "flow")), disturbance=dist2, log_every=12)
razon = simulate(pl, x0, H, DT, C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "ratio"), q_stock_meas=Qs, ratio_nom=R0), disturbance=dist2, log_every=12)
pd.DataFrame({"dilución final [m3/h]": {"consigna fija": 3600*fija.Qd.iloc[-1], "razón fija (FFIC 3102)": 3600*razon.Qd.iloc[-1]},
              "razón dilución/alimentación": {"consigna fija": fija.Qd.iloc[-1]/Qs(H), "razón fija (FFIC 3102)": razon.Qd.iloc[-1]/Qs(H)}}).round(4)

## B4. Etapa 3 · `AIT 3105`: pulso de ClO₄⁻ y retardo del analizador
Pulso de **×1.25 en el ClO₄⁻ de alimentación entre las 3 h y las 5 h**. Como el retardo del analizador aún no está definido, se barre entre 0 y 2 h. Además: válvula de dilución limitada a 3× la nominal frente a una capaz de 8×, y compensación parcial (`gain` = 0.3).

In [ ]:
def pulso(t): return d0["CP_stock"]*(1.25 if 3*3600 <= t < 5*3600 else 1.0)
dist3 = lambda t: {**d0, "CP_stock": pulso(t)}
H = 14*3600
def correr_ait(delay=None, cap=8.0, gain=1.0):
    if delay is None:
        ctrl = C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "flow"))
    else:
        ctrl = C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "ratio", "ait"), q_stock_meas=lambda t: qs0, ratio_nom=R0,
                                 mv_limits={"Q_dil": (0.0, cap*q0)}, ait={"cp_true": pulso, "delay_s": delay, "gain": gain, "R_max": 3*cap*R0})
    return simulate(pl, x0, H, DT, ctrl, disturbance=dist3, log_every=12)

casos = {"sin AIT (dilución fija)": correr_ait(None), "AIT, retardo 0": correr_ait(0.0), "AIT, retardo 15 min": correr_ait(900.0),
         "AIT, retardo 1 h": correr_ait(3600.0), "AIT, retardo 2 h": correr_ait(7200.0),
         "AIT, retardo 0, válvula 3x": correr_ait(0.0, cap=3.0), "AIT, retardo 0, ganancia 0.3": correr_ait(0.0, gain=0.3)}
rows = []
for name, df in casos.items():
    h = df.t_h.values; ppm = df.ppm_ClO4.values; Qd = 3600*df.Qd.values
    rows.append({"caso": name, "ppm pico": ppm.max(), "horas sobre 40 ppm": np.trapezoid((ppm > 40).astype(float), h),
                 "Y mínima %": df.recovery_pct.min(), "Y final %": df.recovery_pct.iloc[-1],
                 "dilución máx [m3/h]": Qd.max(), "agua extra [m3]": np.trapezoid(Qd - 3600*q0, h)})
tabla = pd.DataFrame(rows).set_index("caso"); tabla.round(2)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True)
cols = ["tab:gray", "tab:blue", "tab:cyan", "tab:orange", "tab:red", "tab:purple", "tab:green"]
for (name, df), col in zip(casos.items(), cols):
    ls = "--" if name.startswith("sin") else "-"
    ax[0].plot(df.t_h, df.ppm_ClO4, color=col, ls=ls, label=name)
    ax[1].plot(df.t_h, 3600*df.Qd, color=col, ls=ls)
    ax[2].plot(df.t_h, df.recovery_pct, color=col, ls=ls)
ax[0].axhline(p["spec_ClO4_ppm_max"], color="r", lw=1); ax[0].legend(fontsize=7)
for a in ax: a.axvspan(3, 5, color="orange", alpha=0.12)
ax[0].set(xlabel="t [h]", ylabel="ppm", title="ClO₄⁻ en producto (sombra = pulso en la alimentación)")
ax[1].set(xlabel="t [h]", ylabel="m³/h", title="Agua de dilución (FIRC 3103)"); ax[2].set(xlabel="t [h]", ylabel="%", title="Recuperación de KNO₃")
fig.tight_layout(); fig.savefig("../docs/fig/05_AIT3105_retardo.png"); plt.show()

## Lectura
* **Sin `AIT`:** el pulso apenas supera el límite provisional de 40 ppm, porque el ppm responde lento (la purga arrastra el ClO₄ en horas) y el pulso es corto.
* **Con `AIT` sin retardo:** la compensación total mantiene el ppm, pero **anula la recuperación** mientras dura el pulso y gasta agua adicional del orden de cientos de m³. La compensación total no es una buena ley económica.
* **Retardo:** a 15 min casi no cambia nada; con retardos de una a dos horas (comparables con el pulso) el agua llega cuando el pulso ya pasó: se gasta la misma agua y el ppm pico vuelve a lo del caso sin analizador. **El retardo importa si el pulso es corto frente a él**, por eso hay que definirlo.
* **Válvula de 3×:** limita el daño a la recuperación pero también la compensación.
* **Ganancia parcial:** es la palanca para negociar ppm contra recuperación; su valor debe salir de una evaluación económica (pérdida por producto fuera de norma frente a producción perdida).

**No se modela:** ruido y deriva del analizador, la ubicación exacta del punto de muestreo, tiempos muertos de transporte entre el analizador y la válvula, ni lógica de alarmas (AHL). Faltan por definir: retardo de `AIT 3105`, ganancia y límites de `FFIC 3102`, capacidad real de la válvula de `FIRC 3103`.